# Práctica 2: Comunicaciones y Procesado (Sistemas Embebidos)

**Alumno:** Jordi Blasco Lozano  
**Asignatura:** Sistemas Embebidos  
**Curso:** 2025/2026  
**Fecha:** Marzo de 2026

---

## Resumen ejecutivo

En esta práctica se aborda un flujo completo de **IoT en edge** aplicado al caso de monitorización de pulsaciones con ESP32 y sensor MAX30105. El trabajo integra dos bloques técnicos complementarios:

1. **Procesado de señal en el dispositivo (Parte A):** se limpia y estabiliza la señal de pulso, y se decide *cuándo* conviene transmitir para ahorrar energía y ancho de banda.
2. **Comunicaciones IoT mediante WiFi + MQTT (Parte B):** se conecta el dispositivo a red inalámbrica, se diagnostica su estado de red y se publica telemetría en topics de clase, además de suscribirse a órdenes remotas.

La implementación final combina:

- Lectura de pulso y cálculo de BPM medio.
- Estrategias de envío basadas en **tiempo**, **umbral de cambio** y **alarma**.
- Conectividad WiFi con credenciales separadas.
- Publicación y suscripción MQTT.
- Feedback visual mediante OLED y LED.

El resultado es una solución realista de monitorización biométrica: **envía menos, pero envía mejor**.

## 1. Objetivos oficiales y enfoque de la práctica

### 1.1 Objetivos del enunciado

De acuerdo con el enunciado de **Práctica 2 - Comunicaciones y Procesado**, los objetivos principales son:

- Utilizar la conectividad **WiFi** del ESP32.
- Implementar un cliente **MQTT** para envío de telemetría.
- Aplicar **procesado en el edge** para filtrar datos antes de enviarlos.

### 1.2 Problema técnico que se pretende resolver

Las señales biométricas del pulso suelen incluir ruido por movimiento, contacto irregular del dedo, iluminación ambiente y perturbaciones eléctricas. Si un dispositivo transmite todos los datos en bruto:

- Incrementa el consumo energético (especialmente por radio).
- Aumenta el tráfico innecesario en red.
- Dificulta el análisis en servidor por la baja calidad del dato.

Por ello, en esta práctica se adopta una filosofía de diseño típica en IoT profesional:

- **Procesar localmente** la señal.
- **Detectar eventos relevantes**.
- **Publicar solo información útil**.

### 1.3 Estructura de trabajo seguida

La memoria se organiza en dos partes del enunciado:

- **Parte A (Edge Computing):** estrategias de estabilización y política de envío.
- **Parte B (MQTT):** conectividad, diagnóstico de red y telemetría en broker.

Además, se incluye código completo, evidencias visuales y trazabilidad de cumplimiento de requisitos.

## 2. Arquitectura de la solución implementada

### 2.1 Componentes hardware y software

- **Microcontrolador:** ESP32 (con WiFi integrado).
- **Sensor biométrico:** MAX30105 (lectura de señal IR para detección de latido).
- **Pantalla:** OLED SSD1306 por I2C (estado local y BPM).
- **Actuador:** LED (confirmación de órdenes remotas).
- **Stack de comunicación:** WiFi + MQTT (`PubSubClient`).

### 2.2 Flujo funcional de extremo a extremo

1. El sensor entrega una señal IR que se evalúa para detectar latidos.
2. Se calcula BPM instantáneo y media reciente (`RATE_SIZE = 4`).
3. Se evalúan tres reglas de transmisión:
   - **Regla A:** envío periódico cada 30 s.
   - **Regla B:** envío por cambio significativo (umbral de 5 BPM).
   - **Regla C:** envío por alarma (BPM > 160) con antirrebote temporal de 60 s.
4. Si se cumple alguna condición, se publica en topic MQTT de datos.
5. El OLED muestra BPM y, cuando hay envío, el motivo (A/B/C equivalente).
6. El dispositivo mantiene suscripción a órdenes MQTT para control remoto del LED.

### 2.3 Criterios de diseño adoptados

- **Eficiencia energética:** minimizar transmisiones innecesarias.
- **Fiabilidad del dato:** evitar publicar ruido y oscilaciones irrelevantes.
- **Observabilidad:** mostrar diagnóstico de red y estado de envío en OLED/Serial.
- **Seguridad mínima operativa:** extraer credenciales WiFi a archivo separado (`arduino_secrets.h`).

## 3. Desarrollo de la Parte A: Procesado en el edge

### 3.1 Filtrado y estabilidad de lecturas

El enunciado propone evaluar filtros como SMA, EMA y mediana para estabilizar la señal. En la implementación usada para esta práctica, la estabilización operativa se realiza mediante:

- Detección de latido válida (`checkForBeat(irValue)`), evitando usar muestras sin evento de latido.
- Umbral mínimo de señal IR (`irValue > 50000`) para descartar ausencia de dedo o señal insuficiente.
- Promedio sobre las últimas muestras BPM (`RATE_SIZE = 4`) para suavizar variaciones rápidas.

Aunque no se parametrizan explícitamente funciones separadas `SMA/EMA/Mediana` con nombres propios, el comportamiento final sí aplica una **suavización temporal** efectiva a través de media deslizante corta, suficiente para mejorar estabilidad frente a ruido moderado.

### 3.2 Política de transmisión eficiente (núcleo de la práctica)

Una parte central del trabajo es decidir cuándo transmitir, para equilibrar **consumo**, **ancho de banda** y **relevancia clínica**.

#### Regla A: Downsampling temporal

Se envía un dato cada 30 segundos (`sendInterval = 30000`). Esto asegura telemetría de referencia continua sin sobrecarga.

#### Regla B: Umbral de cambio

Si el BPM actual difiere del último enviado en al menos 5 BPM (`threshold = 5`), se adelanta envío. Con ello se captura dinámica real del paciente sin esperar al periodo fijo.

#### Regla C: Alarma fisiológica

Si `BPM > 160`, se activa envío por condición de alarma. Para evitar flapping en torno al límite, se fuerza un timeout de 60 segundos (`lastAlarmTime`).

### 3.3 Lógica de decisión consolidada

En cada iteración se calculan tres booleanos:

- `timeToSend`
- `thresholdMet`
- `alarmCondition`

Si se cumple cualquiera (`timeToSend || thresholdMet || alarmCondition`), el dispositivo publica el dato y registra el motivo de envío en OLED y Serial.

Este diseño es robusto porque:

- Nunca deja de reportar (gracias a Regla A).
- Reacciona rápido ante cambios relevantes (Regla B).
- Prioriza seguridad ante valores extremos (Regla C).

### 3.4 Visualización local de estado

Durante 2 segundos tras cada envío, el OLED muestra `ENVIADO` + motivo. Esto facilita validación en pruebas de laboratorio porque permite correlacionar:

- Comportamiento de red.
- Valor de BPM enviado.
- Regla que lo ha disparado.

En ausencia de dedo (`irValue <= 50000`), la pantalla indica `Pon el dedo` y el promedio se reinicia, evitando interpretar ruido como pulso válido.

## 4. Desarrollo de la Parte B: Comunicaciones IoT con MQTT

### 4.1 Gestión de credenciales WiFi

Siguiendo el enunciado, las credenciales se ubican en un fichero independiente `arduino_secrets.h`, que se importa en el script principal. Esta práctica evita exponer SSID/clave en repositorios o capturas del código principal.

### 4.2 Diagnóstico de red del ESP32

Tras conexión a WiFi, el dispositivo reporta por monitor serie:

- Dirección IP local.
- Dirección MAC.
- Intensidad de señal RSSI.

Y adicionalmente muestra información en OLED durante unos segundos (`WiFi OK`, IP y RSSI), lo que cumple el apartado extra solicitado en el enunciado.

### 4.3 Conexión MQTT y operación

Se usa `PubSubClient` con broker configurado por IP local:

- Servidor: `192.168.0.106`
- Puerto: `1883`

En producción de prácticas se podría alternar con broker público (`broker.hivemq.com` o `test.mosquitto.org`) o broker local con Docker, pero el flujo implementado ya valida el comportamiento completo publicación/suscripción.

### 4.4 Topics utilizados

- Publicación de pulsaciones: `clase/salud/MV/datos`
- Suscripción a órdenes: `clase/salud/MV/ordenes`

El payload publicado es el BPM numérico en texto, siguiendo el formato esperado por el enunciado. En órdenes, un `'1'` enciende LED y otro valor lo apaga.

### 4.5 Robustez de conexión

Se incorpora función `reconnect()` para reconectar automáticamente al broker cuando se pierde sesión MQTT, con `clientId` aleatorio para reducir colisiones de identidad en reconexiones.

### 4.6 Relación entre Parte A y Parte B

La Parte A selecciona **qué dato merece enviarse** y la Parte B implementa **cómo enviarlo** de forma interoperable. Esta separación de responsabilidades es clave en sistemas embebidos conectados:

- Bloque de señal = calidad del dato.
- Bloque de red = entrega del dato.

## 5. Código completo del script principal (`prueba_sin.ino`)

A continuación se incluye el código íntegro empleado en la práctica. Se mantiene en un único bloque para facilitar trazabilidad entre memoria y entrega.

```c
#include <WiFi.h>
#include <Wire.h>
#include <Adafruit_GFX.h>
#include <Adafruit_SSD1306.h>
#include "MAX30105.h"
#include "heartRate.h"
#include <PubSubClient.h>
#include "arduino_secrets.h"

// --- PINES ---
#define SDA_PIN 5
#define SCL_PIN 6
#define LED_R 7

// --- CONFIGURACION OLED ---
#define SCREEN_WIDTH 128
#define SCREEN_HEIGHT 32
#define OLED_RESET -1
Adafruit_SSD1306 display(SCREEN_WIDTH, SCREEN_HEIGHT, &Wire, OLED_RESET);

// --- CONFIGURACION SENSOR Y FILTRO ---
MAX30105 particleSensor;
const byte RATE_SIZE = 4;
long rates[RATE_SIZE];
byte rateSpot = 0;
long lastBeat = 0;
float beatsPerMinute;
int beatAvg = 0;

// --- VARIABLES PARTE A (BATERIA) ---
unsigned long lastSendTime = 0;
int lastSentValue = 0;
const int threshold = 5;
const unsigned long sendInterval = 30000;
unsigned long lastAlarmTime = 0;

// Variables para el mensaje visual en pantalla
unsigned long tiempoAvisoOLED = 0;
String motivoAviso = "";

// --- MQTT ---
const char* mqtt_server = "192.168.0.106";
WiFiClient espClient;
PubSubClient client(espClient);

void setup_wifi_and_diagnostics() {
  WiFi.begin(SECRET_SSID, SECRET_PASS);
  while (WiFi.status() != WL_CONNECTED) {
    delay(500);
    Serial.print(".");
  }

  Serial.println("\n--- Diagnostico de Red ---");
  Serial.print("IP: "); Serial.println(WiFi.localIP());
  Serial.print("MAC: "); Serial.println(WiFi.macAddress());
  Serial.print("RSSI: "); Serial.print(WiFi.RSSI()); Serial.println(" dBm");

  display.clearDisplay();
  display.setTextSize(1);
  display.setCursor(0,0);
  display.println("WiFi OK!");
  display.print("IP: "); display.println(WiFi.localIP());
  display.print("RSSI: "); display.print(WiFi.RSSI()); display.println("dBm");
  display.display();
  delay(3000);
}

void callback(char* topic, uint8_t* payload, unsigned int length) {
  if ((char)payload[0] == '1') {
    digitalWrite(LED_R, HIGH);
  } else {
    digitalWrite(LED_R, LOW);
  }
}

void reconnect() {
  while (!client.connected()) {
    String clientId = "ESP32_MV_" + String(random(0xffff), HEX);
    if (client.connect(clientId.c_str())) {
      client.subscribe("clase/salud/MV/ordenes");
    } else {
      delay(5000);
    }
  }
}

void setup() {
  Serial.begin(115200);

  Wire.begin(SDA_PIN, SCL_PIN);

  pinMode(LED_R, OUTPUT);
  digitalWrite(LED_R, LOW);

  if(!display.begin(SSD1306_SWITCHCAPVCC, 0x3C)) {
    Serial.println("Error OLED");
    for(;;);
  }
  display.clearDisplay();
  display.setTextColor(SSD1306_WHITE);

  setup_wifi_and_diagnostics();

  if (!particleSensor.begin(Wire, I2C_SPEED_FAST)) {
    Serial.println("Error MAX30105");
    display.clearDisplay();
    display.setCursor(0,0);
    display.print("Sensor no detectado");
    display.display();
    for(;;);
  }
  particleSensor.setup();
  particleSensor.setPulseAmplitudeRed(0x0A);

  client.setServer(mqtt_server, 1883);
  client.setCallback(callback);
}

void loop() {
  if (!client.connected()) reconnect();
  client.loop();

  long irValue = particleSensor.getIR();

  if (checkForBeat(irValue) == true && irValue > 50000) {
    long delta = millis() - lastBeat;
    lastBeat = millis();
    beatsPerMinute = 60 / (delta / 1000.0);

    if (beatsPerMinute < 255 && beatsPerMinute > 20) {
      rates[rateSpot++] = (long)beatsPerMinute;
      rateSpot %= RATE_SIZE;
      beatAvg = 0;
      for (byte x = 0 ; x < RATE_SIZE ; x++) beatAvg += rates[x];
      beatAvg /= RATE_SIZE;
    }
  }

  unsigned long currentTime = millis();

  bool alarmCondition = (beatAvg > 160 && (currentTime - lastAlarmTime > 60000));
  bool timeToSend = (currentTime - lastSendTime > sendInterval);
  bool thresholdMet = (abs(beatAvg - lastSentValue) >= threshold);

  if (irValue > 50000) {

    // Si se cumple alguna regla, disparamos el envio
    if (timeToSend || thresholdMet || alarmCondition) {
      char msg[10];
      dtostrf(beatAvg, 1, 0, msg);

      client.publish("clase/salud/MV/datos", msg);

      lastSentValue = beatAvg;
      lastSendTime = currentTime;
      if (alarmCondition) lastAlarmTime = currentTime;

      // Guardamos el motivo y el tiempo para que se muestre en pantalla
      tiempoAvisoOLED = currentTime;
      if (alarmCondition) motivoAviso = "Regla C (Alarma)";
      else if (thresholdMet) motivoAviso = "Regla B (Umbral)";
      else motivoAviso = "Regla A (Tiempo)";

      // Aviso claro por el Monitor Serie
      Serial.println("\n==================================");
      Serial.print(">> ENVIANDO PULSACIONES: ");
      Serial.print(beatAvg);
      Serial.println(" BPM");
      Serial.print(">> MOTIVO: ");
      Serial.println(motivoAviso);
      Serial.println("==================================\n");
    }

    display.clearDisplay();

    // Mostrar el pulso actual
    display.setTextSize(2);
    display.setCursor(0, 0);
    display.print("BPM: "); display.print(beatAvg);

    // Si hace menos de 2000 milisegundos que enviamos el dato, mostrar el mensaje
    if (currentTime - tiempoAvisoOLED < 2000) {
      display.setTextSize(1);
      display.setCursor(0, 20);
      display.print("ENVIADO: ");
      display.print(motivoAviso);
    }

    display.display();

  } else {
    display.clearDisplay();
    display.setTextSize(2);
    display.setCursor(0, 10);
    display.print("Pon el dedo");
    display.display();
    beatAvg = 0;
  }
}
```

### 5.1 Comentario técnico del código

Este script integra en un único ciclo principal la adquisición de señal, cálculo de BPM medio, toma de decisión de envío y publicación MQTT. Es una solución compacta y eficaz para práctica de laboratorio, y reproduce bien un caso de uso real de telemetría biométrica embebida.

## 6. Código de credenciales (`arduino_secrets.h`)

Se incluye por completitud documental y porque forma parte explícita del enunciado (gestión separada de secretos).

```c
#ifndef ARDUINO_SECRETS_H
#define ARDUINO_SECRETS_H

#define SECRET_SSID "TP-Link_7BC5"
#define SECRET_PASS "81912834"

#endif
```

### 6.1 Nota de buenas prácticas

Para entregas públicas o repositorios abiertos, este archivo debería ir en `.gitignore` y usarse una plantilla tipo `arduino_secrets.example.h` sin credenciales reales.

## 7. Evidencias visuales y análisis de resultados

En esta sección se incorporan todas las imágenes disponibles de la práctica, con interpretación técnica de cada evidencia.

### 7.1 Diagnóstico de red y conexión

**Imagen:** Comprobación de WiFi y estado de red.

![Comprobación WiFi y diagnóstico](fotos/Comprobacion_wifi_diagnostico_red.png)

**Análisis:**

- Confirma asociación correcta a red inalámbrica.
- Permite verificar IP asignada y calidad de enlace (RSSI).
- Es crítica para aislar fallos de conectividad antes de depurar MQTT.

### 7.2 Diagrama de referencia del montaje

**Imagen:** Esquema/diagrama usado en la práctica.

![Diagrama a usar](fotos/diagrama_a_usar.png)

**Análisis:**

- Sirve de trazabilidad entre diseño conceptual y montaje efectivo.
- Reduce errores de cableado I2C y alimentación.
- Facilita reproducibilidad para evaluación docente.

### 7.3 Disparo por regla B (umbral)

**Imagen:** Evidencia de envío por variación significativa.

![Motivo Regla B umbral](fotos/motivo_regla_B_umbral.png)

**Análisis:**

- Demuestra que el sistema no depende únicamente del temporizador.
- Verifica el comportamiento reactivo ante cambios de pulso.
- Confirma visualmente la etiqueta de motivo de envío.

### 7.4 Captura de prueba con valor de pulsaciones

**Imagen:** Registro de valor de pulsaciones durante monitorización.

![Captura de 31 pulsaciones](fotos/prueba_captura_de_pantalla_31_pulsaciones.png)

**Análisis:**

- Aporta evidencia de adquisición y visualización en tiempo real.
- Es útil para correlacionar lecturas con estado del dedo/sensor.
- Sirve para validar que el pipeline completo está operativo.

### 7.5 Prueba en entorno de clase/proyector

**Imagen:** Demostración de envío/recepción en infraestructura de laboratorio.

![Prueba enviando pulsaciones en proyector](fotos/prueba_enviando_pulsaciones_proyector_de_clase.jpeg)

**Análisis:**

- Evidencia interoperabilidad con el ecosistema de prácticas.
- Demuestra que los datos llegan al panel o cliente de monitorización.
- Refuerza validación de topics y formato de payload.

### 7.6 Prueba con dedo real sobre el sensor

**Imagen:** Captura de uso real con contacto físico.

![Prueba con dedo real](fotos/pulsaciones_prueba_dedo_real.jpeg)

**Análisis:**

- Verifica comportamiento en condición real de uso.
- Permite observar robustez frente a pequeños movimientos.
- Demuestra que el sistema no se limita a valores simulados.

### 7.7 Conclusión de resultados experimentales

Las evidencias muestran que el sistema:

- Se conecta correctamente por WiFi.
- Publica telemetría MQTT de forma controlada.
- Diferencia motivos de envío según estrategia A/B/C.
- Funciona tanto en ensayo local como en entorno de clase.
- Proporciona feedback inmediato en OLED y monitor serie.

## 8. Trazabilidad de cumplimiento respecto al enunciado

### 8.1 Parte A: Procesado

- **Estabilización de lecturas:** implementada mediante validación de latido + media reciente (`RATE_SIZE = 4`).
- **Downsampling (30 s):** implementado con `sendInterval = 30000`.
- **Umbral de cambio (5 BPM):** implementado con `threshold = 5`.
- **Alarma BPM > 160 con timeout de 1 min:** implementada con `alarmCondition` y `lastAlarmTime`.
- **Mostrar motivo de envío:** implementado y visible en OLED/Serial (`Regla A`, `Regla B`, `Regla C`).

### 8.2 Parte B: Comunicación

- **Credenciales en archivo independiente:** `arduino_secrets.h` incluido y usado.
- **Diagnóstico de red del dispositivo:** IP, MAC y RSSI por Serial + OLED.
- **Cliente MQTT operativo:** conexión, publicación y suscripción funcionales.
- **Telemetría en topic de datos:** envío de BPM numérico al topic definido.
- **Recepción de órdenes:** suscripción a topic de órdenes y actuación sobre LED.

### 8.3 Observaciones técnicas

- El valor de gateway no se imprime explícitamente en el código actual. Si se desea cubrir ese punto al 100%, basta añadir: `Serial.println(WiFi.gatewayIP());`.
- La parte de filtros SMA/EMA/mediana puede extenderse con implementaciones diferenciadas para comparación cuantitativa en futuras iteraciones.

## 9. Conclusiones y propuestas de mejora

### 9.1 Conclusiones

La práctica cumple el objetivo principal de transformar un prototipo de lectura biométrica en un **nodo IoT eficiente**, capaz de:

- Filtrar y decidir localmente.
- Comunicar solo eventos relevantes.
- Mantener telemetría periódica de respaldo.
- Integrarse en flujo MQTT de laboratorio.

Desde el punto de vista de ingeniería embebida, el sistema resultante demuestra un diseño correcto de compromiso entre **precisión**, **coste energético** y **conectividad**.

### 9.2 Mejoras recomendadas

1. Separar módulos en archivos (`sensor.cpp`, `mqtt.cpp`, `display.cpp`) para mejorar mantenibilidad.
2. Añadir comparación explícita de filtros (SMA/EMA/mediana) con métricas de varianza y latencia.
3. Incorporar histéresis de alarma avanzada (umbral alto/umbral bajo) además del timeout.
4. Migrar a MQTT con TLS en entornos no académicos.
5. Persistir eventos críticos en memoria local por si se pierde conectividad.

---

## Anexo: comando de despliegue opcional de broker local

Si se opta por alternativa local con Docker (según enunciado):

```c
docker-compose up -d
```

Después se puede acceder a paneles como `http://127.0.0.1:1880/` y configurar el ESP32 apuntando a la IP local del equipo host.